# PE-AV Embedding Extraction

Extract audio, video, and joint (AV) embeddings from PE-AV model variants.
Configure `MODEL_IDS` below to run one or multiple variants.

**Environment:** `avtransformer`  
**Output:** `/outputs/model_embeddings/{model_name}/{bin_sec}s/{model_name}_{v,a,av}.npy`

In [ ]:
import gc
import os
import re
from pathlib import Path

import numpy as np
import torch
from natsort import natsorted
from transformers import AutoModel, AutoProcessor

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Add or remove entries to control which model variants are processed.
# hf_id:      HuggingFace model identifier used for from_pretrained()
# modalities: which output keys to save — subset of ["v", "a", "av"]
MODEL_CONFIGS = {
    "pe-av-small-16-frame": {
        "hf_id": "facebook/pe-av-small-16-frame",
        "modalities": ["v", "a", "av"],
    },
    "pe-av-small": {
        "hf_id": "facebook/pe-av-small",
        "modalities": ["v", "a", "av"],
    },
    "pe-av-base": {
        "hf_id": "facebook/pe-av-base",
        "modalities": ["v", "a", "av"],
    },
    "pe-av-base-16-frame": {
        "hf_id": "facebook/pe-av-base-16-frame",
        "modalities": ["v", "a", "av"],
    },
    "pe-av-large": {
        "hf_id": "facebook/pe-av-large",
        "modalities": ["v", "a", "av"],
    },
    "pe-av-large-16-frame": {
        "hf_id": "facebook/pe-av-large-16-frame",
        "modalities": ["v", "a", "av"],
    },
}

SEGMENT_DURATIONS = [1.0, 2.0, 5.0, 10.0]  # seconds
BATCH_SIZE = 8
LOCAL_FILES_ONLY = True

DATA_BASE = Path("/home/amin/Research/Representation/Movie/data/Setareh/Data")
RUNS_BASE = Path("/home/amin/Research/Representation/Movie/data/Setareh/runs")
OUTPUT_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
# ─────────────────────────────────────────────────────────────────────────────

## Helper Functions

In [ ]:
def get_model_shortname(model_id: str) -> str:
    """Convert HuggingFace model ID to a short filesystem-safe name."""
    return model_id.split("/")[-1]


def find_chunk_pairs(data_base: Path, seg_sec: float) -> list[tuple[Path, Path]]:
    """Find all paired (video .mp4, audio .wav) chunks for a given segment duration."""
    pairs = []
    chunk_dirs = natsorted(data_base.glob(f"*_chunks_{int(seg_sec)}s"))
    for chunk_dir in chunk_dirs:
        videos = natsorted(chunk_dir.glob("*_part_*.mp4"))
        for vid in videos:
            part_num = re.search(r"_part_(\d+)", vid.stem).group(1)
            parent_stem = re.sub(r"_chunks_\d+s$", "", chunk_dir.name)
            aud = chunk_dir / f"{parent_stem}_part_{part_num}.wav"
            if not aud.exists():
                aud = chunk_dir / f"{parent_stem}_part_{part_num}.m4a"
            if aud.exists():
                pairs.append((vid, aud))
    return pairs


@torch.no_grad()
def extract_embeddings(model, processor, pairs: list, batch_size: int,
                        device: str) -> dict[str, np.ndarray]:
    """Extract audio, video, and joint embeddings from video/audio file pairs.

    Returns dict with keys 'v', 'a', 'av' each containing (N, D) arrays.
    """
    video_embeds, audio_embeds, av_embeds = [], [], []

    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        vid_paths = [str(p[0]) for p in batch]
        aud_paths = [str(p[1]) for p in batch]

        inputs = processor(videos=vid_paths, audios=aud_paths, return_tensors="pt")
        inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                  for k, v in inputs.items()}

        outputs = model(**inputs)
        video_embeds.append(outputs.video_embeds.cpu().numpy())
        audio_embeds.append(outputs.audio_embeds.cpu().numpy())
        av_embeds.append(outputs.audio_video_embeds.cpu().numpy())

        if (i // batch_size) % 10 == 0:
            print(f"    {i + len(batch)}/{len(pairs)} chunks processed", end="\r")

    print()
    return {
        "v":  np.concatenate(video_embeds, axis=0),
        "a":  np.concatenate(audio_embeds, axis=0),
        "av": np.concatenate(av_embeds, axis=0),
    }

## Data Compatibility Check

Verify paired video/audio chunks exist for each segment duration.

In [ ]:
print("Chunk pair counts per segment duration:")
for seg_sec in SEGMENT_DURATIONS:
    pairs = find_chunk_pairs(DATA_BASE, seg_sec)
    print(f"  {seg_sec:4.1f}s → {len(pairs):4d} pairs")

## Embedding Extraction

Loop over all models in `MODEL_IDS`. Each model is loaded, all segment durations
are processed, and the model is deleted to free GPU memory before the next iteration.

In [ ]:
for model_name, cfg in MODEL_CONFIGS.items():
    model_id = cfg["hf_id"]
    modalities = cfg["modalities"]
    print(f"\n{'='*60}")
    print(f"Model: {model_id}")
    print(f"{'='*60}")

    # Load model
    processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
    model = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
    model.eval()

    for seg_sec in SEGMENT_DURATIONS:
        out_dir = OUTPUT_BASE / model_name / f"{int(seg_sec)}s"
        out_dir.mkdir(parents=True, exist_ok=True)

        # Skip if all modality files already exist
        if all((out_dir / f"{model_name}_{m}.npy").exists() for m in modalities):
            print(f"  {seg_sec}s — already done, skipping")
            continue

        print(f"  {seg_sec}s — collecting pairs ...")
        pairs = find_chunk_pairs(DATA_BASE, seg_sec)
        print(f"  {seg_sec}s — extracting from {len(pairs)} pairs ...")

        embeds = extract_embeddings(model, processor, pairs, BATCH_SIZE, DEVICE)

        for mod, arr in embeds.items():
            if mod not in modalities:
                continue
            out_path = out_dir / f"{model_name}_{mod}.npy"
            np.save(out_path, arr)
            print(f"    saved {out_path.name}  shape={arr.shape}")

    # Release GPU memory before loading next model
    del model, processor
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  {model_name} complete, GPU memory released")

print("\nAll models done.")

## Full Movie Runs (Optional)

Extract embeddings from full-length movie runs (not pre-chunked segments).
Outputs are saved under `{model}/{bin_sec}s/full/`.

In [ ]:
EXTRACT_FULL_RUNS = False  # Set to True to run this cell

if EXTRACT_FULL_RUNS:
    for model_id in MODEL_IDS:
        short = get_model_shortname(model_id)
        processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
        model = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
        model.eval()

        for seg_sec in SEGMENT_DURATIONS:
            run_files = natsorted(RUNS_BASE.glob("*.mp4"))
            pairs = [(f, f.with_suffix(".wav")) for f in run_files
                     if f.with_suffix(".wav").exists()]

            out_dir = OUTPUT_BASE / short / f"{int(seg_sec)}s" / "full"
            out_dir.mkdir(parents=True, exist_ok=True)

            embeds = extract_embeddings(model, processor, pairs, BATCH_SIZE, DEVICE)
            for mod, arr in embeds.items():
                np.save(out_dir / f"{short}_{mod}.npy", arr)

        del model, processor
        torch.cuda.empty_cache()
        gc.collect()